In [1]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
import calendar
from datetime import datetime  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta
from collections import defaultdict
from dotenv import load_dotenv  


warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [18]:
def read_protected_excel(file_path, password):  
    """Return {sheet_name: DataFrame} from a password-protected workbook."""  
    decrypted = io.BytesIO()  
    with open(file_path, 'rb') as f:  
        office_file = msoffcrypto.OfficeFile(f)  
        if office_file.is_encrypted():  
            office_file.load_key(password=password)  
            office_file.decrypt(decrypted)  
        else:  
            decrypted.write(f.read())   # handle any unprotected files  
    decrypted.seek(0)  
    # sheet_name=None -> dict of every tab  
    return pd.read_excel(decrypted, sheet_name=None)

load_dotenv() 
base_path = r'L:/2026 Pilar Plant Files/AP EPIC Audit/LENOX/Backup/EPIC - Received from Meaghan'  
password = os.environ['EPIC_XL_PW'] 

epic_files = [  
    p for p in glob.glob(os.path.join(base_path, '*.xls*'))  
    if not os.path.basename(p).startswith('~$')  
]

by_sheet = defaultdict(list)

for file_path in epic_files:  
    try:  
        sheets = read_protected_excel(file_path, password)  
        for sheet_name, df in sheets.items():  
            df['source_file'] = os.path.basename(file_path)  
            df['source_sheet'] = sheet_name  
            by_sheet[sheet_name].append(df)  
        print(f"OK: {os.path.basename(file_path)} ({len(sheets)} tabs)")  
    except Exception as e:  
        print(f"ERROR in {os.path.basename(file_path)}: {e}")

# one combined dataframe per tab name  
epic_by_sheet = {name: pd.concat(frames, ignore_index=True)  
                 for name, frames in by_sheet.items()}

# or everything stacked into one, if the tabs share a schema  
epic_data_all = (pd.concat(epic_by_sheet.values(), ignore_index=True)  
                 if epic_by_sheet else pd.DataFrame())

print(f"\nTabs found: {list(epic_by_sheet)}")  
for name, df in epic_by_sheet.items():  
    print(f"  {name}: {df.shape[0]:,} rows x {df.shape[1]} cols")

OK: LHH August 2026 Implants.xlsx (1 tabs)
OK: LHH July 2026 Implants.xlsx (1 tabs)
OK: LHH Nov - Jun 2026 FULL.xlsx (1 tabs)

Tabs found: ['Sheet1']
  Sheet1: 119,498 rows x 24 cols


In [3]:
display(epic_data_all.head())

,Location,Date,MRN,CSN,Case #,Name,Imp PS ID,Manufacturer,Lot #,Model #,...,Action,Implant Reason Wasted,Cost,Implant Used Cost,Implant Wasted Cost,Implant Total Cost,Number Used,Primary Cvg Payer,source_file,source_sheet
0,LHH CATH & EP LABS,2026-08-07,30481942,160063113964,637425.0,Lead Solia S 53 Pm - S8002536388 - Log637425 [...,228580.0,BIOTRONIK GMBH & CO [532],NaN,377177,...,Implanted,NaN,340.0,7462.48,0.0,7462.48,1,HEALTHFIRST,LHH August 2026 Implants.xlsx,Sheet1
1,LHH CATH & EP LABS,2026-08-07,30481942,160063113964,637425.0,Envelope Tyrx Absorbable Antibacterial Med - L...,195910.0,MEDTRONIC CRM [45],R275366,CMRM6122,...,Implanted,NaN,995.0,7462.48,0.0,7462.48,1,HEALTHFIRST,LHH August 2026 Implants.xlsx,Sheet1
2,LHH CATH & EP LABS,2026-08-07,30481942,160063113964,637425.0,Pacemaker Amvia Edge Dr-T - S1000903757 - Log6...,293230.0,BIOTRONIK GMBH & CO [532],NaN,460163,...,Implanted,NaN,4335.0,7462.48,0.0,7462.48,1,HEALTHFIRST,LHH August 2026 Implants.xlsx,Sheet1
3,LHH CATH & EP LABS,2026-08-07,24863511,160063237111,639990.0,Micro-Intro 5f 0.018x40cm Nitinol Ss Tip 7cm E...,236190.0,TELEFLEX VASCULAR SOLN ARROW [554],73M2400567,7200V,...,Implanted and Removed,NaN,29.5,14598.48,0.0,14598.48,1,HEALTHFIRST MEDICARE,LHH August 2026 Implants.xlsx,Sheet1
4,LHH CATH & EP LABS,2026-08-07,24863511,160063237111,639990.0,Sheath Introducer Terumo Pinnacle Coronary 6fr...,5640.0,TERUMO CORP TERUMO MED CORP [31],NaN,RSS602,...,Implanted and Removed,NaN,11.0,14598.48,0.0,14598.48,1,HEALTHFIRST MEDICARE,LHH August 2026 Implants.xlsx,Sheet1


In [13]:
ap_path = r'L:/2026 Pilar Plant Files/AP EPIC Audit/LENOX/Backup/AP'  

# get all matching files    
pc_files = glob.glob(os.path.join(ap_path, f'*.xlsx'))

print(f"Found {len(pc_files)} files")

ap_data = []

for file in pc_files:    
    filename = os.path.basename(file)    
      
    try:  
        # read second tab, skip first 3 rows    
        file_data = pd.read_excel(file, sheet_name='Voucher Drill', skiprows=3)    
          
        ap_data.append(file_data)  
        print(f"Processed: {filename}")  
      
    except PermissionError:  
        print(f"SKIPPED (permission denied): {filename}")  
        continue

# combine all into one dataframe    
if ap_data:    
    ap_data_all = pd.concat(ap_data, ignore_index=True)    
    print(f"\nTotal rows: {len(position_control)}")    
else:    
    print(f"No files found")  

Found 3 files
Processed: Lenox voucher detail 11 2025.xlsx
Processed: Lenox voucher detail 12 2025.xlsx
Processed: Lenox voucher detail YTD 9 2026.xlsx

Total rows: 85104


In [20]:
depts_xwalk = pd.read_excel('L:/2026 Pilar Plant Files/Crosswalks/Manhattan Region Crosswalk.xlsx', sheet_name='departments')
acct_xwalk = pd.read_excel('L:/2026 Pilar Plant Files/Crosswalks/Manhattan Region Crosswalk.xlsx', sheet_name='all_accounts')

display(depts_xwalk)
display(acct_xwalk)

,BU,Dept,Dept Name
0,LENOX,15600000,Hospital Administration
1,LENOX,15600020,Pastoral Care
2,LENOX,15600030,Finance Site
3,LENOX,15600050,Safety
4,LENOX,15600055,Consulting Services
...,...,...,...
543,GOLDN,73009125,Food and Nutrition
544,GOLDN,73009126,Lobby Boutique
545,GOLDN,73009197,Emergency Events 1
546,GOLDN,73009198,Emergency Events 2


,Account,Description
0,10050,Vehicles
1,10800,Cash-Operating
2,11140,Investments
3,12000,Gross Ar - Inpatient
4,12010,Cash Control
...,...,...
516,65063,Gas-LeaseProperty Var Pd to LL
517,65054,Other telephone service
518,63309,Rx Supplemental Pharmacy
519,66527,Equipment Rentals SI


In [28]:
epic_data_raw = epic_data_all[['Location', 'Date', 'Name', 'Imp PS ID', 'Manufacturer', 'Cost', 'Number Used']]

display(epic_data_raw)

,Location,Date,Name,Imp PS ID,Manufacturer,Cost,Number Used
0,LHH CATH & EP LABS,2026-08-07,Lead Solia S 53 Pm - S8002536388 - Log637425 [...,228580.0,BIOTRONIK GMBH & CO [532],340.0,1
1,LHH CATH & EP LABS,2026-08-07,Envelope Tyrx Absorbable Antibacterial Med - L...,195910.0,MEDTRONIC CRM [45],995.0,1
2,LHH CATH & EP LABS,2026-08-07,Pacemaker Amvia Edge Dr-T - S1000903757 - Log6...,293230.0,BIOTRONIK GMBH & CO [532],4335.0,1
3,LHH CATH & EP LABS,2026-08-07,Micro-Intro 5f 0.018x40cm Nitinol Ss Tip 7cm E...,236190.0,TELEFLEX VASCULAR SOLN ARROW [554],29.5,1
4,LHH CATH & EP LABS,2026-08-07,Sheath Introducer Terumo Pinnacle Coronary 6fr...,5640.0,TERUMO CORP TERUMO MED CORP [31],11.0,1
...,...,...,...,...,...,...,...
119493,LHH OR,2026-04-17,Graft I-Factor Putty 5cc - Sn/A - Log291854 [2...,255250.0,CERA PRDTS LLC [1352],2800.0,1
119494,LHH OR,2026-04-17,Screw Vader Pedicle 5.5x30mm - Sn/A - Log29185...,278310.0,ICOTEC MEDICAL INC [1464],4175.0,1
119495,LHH OR,2026-03-04,Sheath Intro Dryseal Flex 14fr 33cm - S1000004...,233660.0,WL GORE & ASSC INC [105],543.0,1
119496,LHH OR,2026-03-02,Screw Maxdrive 1 Lvl 2x7mm - S258790791 - Log1...,165320.0,KLS MARTIN LP [300],49.6,2


In [55]:


ap_data_spec = ap_data_all[['Source Type', 'GL Unit', 'Year', 'Period', 'Account', 'Dept', 'Monetary Amount', 'Vendor ID', 'Vendor Name', 'Qty', 'UOM', 'Unit Price', 'Item ID', 'Description']]

ap_data_dept = pd.merge(ap_data_spec, depts_xwalk[['Dept', 'Dept Name']], how='left', on='Dept')
ap_data_acct = pd.merge(ap_data_dept, acct_xwalk, how='left', on='Account')
ap_data_full = ap_data_acct.rename(columns={'Description_x': 'Item Desc', 'Description_y': 'Acct Desc'})

ap_accounts = ap_data_full[['Account', 'Acct Desc', 'Item ID']].dropna().drop_duplicates().reset_index(drop=True)

itemids = ap_accounts.groupby(['Item ID'])[['Item ID']].count().reset_index(drop=True).sort_values(by='Item ID', ascending=False)
display(itemids)


,Item ID
4972,5
7310,5
4865,4
3861,4
3428,4
...,...
7576,1
7577,1
7578,1
7579,1
